# 2. Vector Damage Assessment

This notebook demonstrates flood damage assessment using vector exposure data (polygons) with a raster hazard map.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from damagescanner import DamageScanner

## Define input data

- **Hazard**: Flood inundation map (GeoTIFF)
- **Exposure**: Land use polygons (GeoPackage)
- **Vulnerability**: Depth-damage curves and maximum damage values

In [ ]:
data_path = Path("..") / "data" / "kampen"

hazard = data_path / "hazard" / "1in100_inundation_map.tif"
exposure = data_path / "exposure" / "landuse.gpkg"
curves = data_path / "vulnerability" / "curves_landuse.csv"
maxdam = data_path / "vulnerability" / "maxdam_landuse.csv"

## Initialize DamageScanner

The scanner detects vector exposure data and uses the vector-based assessment approach.

In [ ]:
ds = DamageScanner(
    hazard_data=hazard,
    feature_data=exposure,
    curves=curves,
    maxdam=maxdam,
)

print(f"Assessment type: {ds.assessment_type}")

## Calculate damages

The `object_col` parameter specifies which column contains the land use classification.

In [ ]:
result = ds.calculate(object_col="landuse")

print(f"Features assessed: {len(result)}")
print(f"Total damage: €{result['damage'].sum():,.0f}")
result.head()

## Summarize by land use type

In [ ]:
damage_by_landuse = (
    result.groupby("landuse")["damage"].sum().sort_values(ascending=False)
)
damage_by_landuse

## Visualize results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Map of damages
result.plot(column="damage", cmap="Reds", legend=True, ax=axes[0])
axes[0].set_title("Damage per Feature")

# Bar chart
damage_by_landuse.plot.bar(ax=axes[1])
axes[1].set_title("Damage by Land Use Type")
axes[1].set_ylabel("Damage (€)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()